# cloc Comment-to-Code Ratio — Raw Output Extraction (C)

This notebook analyzes **C repositories** with **cloc (Count Lines of Code)** and captures complete raw tool output for Comment-to-Code Ratio, blank lines, comment lines, and code lines.

**Default benchmark repository:** [redis/redis](https://github.com/redis/redis)

> **Note:** Comment-to-Code Ratio is a **Derived** metric computed from cloc output (comment / code). cloc does not emit this ratio directly.


## Section 1 — Install Dependencies

Install Python packages and verify cloc. If cloc is unavailable via pip, use the OS package manager (e.g. `sudo apt-get install -y cloc` on Ubuntu).


In [ ]:
!pip install -q pandas gitpython jupyter

import shutil, subprocess, sys

try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cloc'], check=False)
except Exception:
    pass

if not shutil.which('cloc'):
    print('cloc not on PATH; notebook will bootstrap to ../../runtimes/cloc/ if needed.')
else:
    subprocess.run(['cloc', '--version'], check=False)


## Section 2 — Configuration


In [ ]:
USE_GIT_URL = True

REPO_URL = 'https://github.com/redis/redis.git'

LOCAL_REPO_PATH = '/content/redis'

WORKSPACE_DIR = './workspace'

OUTPUT_DIR = './outputs'

IF_CLONE_EXISTS = 'reuse'

CLONE_DEPTH = 1

RAW_OUTPUT_PREVIEW_LINES = 150

from pathlib import Path

METRIC_ROOT = Path('.').resolve()
PROJECT_ROOT = METRIC_ROOT
for _ in range(8):
    if (PROJECT_ROOT / 'runtimes').is_dir() or (PROJECT_ROOT / 'README.md').is_file():
        break
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
RUNTIMES_ROOT = PROJECT_ROOT / 'runtimes'

# Fast validation benchmark:
# USE_GIT_URL = False
# LOCAL_REPO_PATH = './workspace/comment_to_code_ratio_benchmark'


## Section 3 — Imports and Utility Functions


In [ ]:
from pathlib import Path

from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
import urllib.request
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError

C_FILE_EXTENSIONS = {".c", ".h"}
EXCLUDED_DIR_NAMES = {".git", "build", "dist", "bin", "vendor", "docs", "tests", "third_party"}
CLOC_EXCLUDE_DIRS = "build,dist,bin,vendor,docs,tests,third_party,.git"
CLOC_VERSION = "2.08"
CLOC_RELEASE_TAG = "v2.08"
CLOC_WINDOWS_URL = f"https://github.com/AlDanial/cloc/releases/download/{CLOC_RELEASE_TAG}/cloc-{CLOC_VERSION}.exe"
CLOC_PERL_URL = f"https://github.com/AlDanial/cloc/releases/download/{CLOC_RELEASE_TAG}/cloc-{CLOC_VERSION}.pl"
METRICS_COLUMNS = ["language", "files", "blank_lines", "comment_lines", "code_lines"]


class NotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        self._errors: list[dict[str, str]] = []
        if not self.error_log_path.exists():
            self.write_errors()

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str, file: str = "notebook") -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        self._errors.append({"timestamp": timestamp, "file": file, "error_message": message})
        self.write_errors()

    def write_errors(self) -> None:
        with self.error_log_path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=["timestamp", "file", "error_message"])
            writer.writeheader()
            writer.writerows(self._errors)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str, workspace_dir: Path, if_clone_exists: str, logger: NotebookLogger, clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)
    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError("IF_CLONE_EXISTS must be 'reuse' or 'reclone'.")
    logger.info(f"Cloning {repo_url} into {clone_path}")
    clone_kwargs: dict[str, Any] = {"depth": clone_depth} if clone_depth else {}
    try:
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Git clone failed: {exc}", file=repo_url)
        raise
    return clone_path.resolve()


def validate_local_repo_path(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        msg = f"Local repository path does not exist: {local_repo_path}"
        logger.error(msg, file=str(local_repo_path))
        raise FileNotFoundError(msg)
    if not local_repo_path.is_dir():
        msg = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(msg, file=str(local_repo_path))
        raise NotADirectoryError(msg)
    try:
        Repo(local_repo_path)
        logger.info("Validated Git repository.")
    except InvalidGitRepositoryError:
        logger.info("Path is not a Git repository; proceeding as source directory.")
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_url: bool, repo_url: str, local_repo_path: str | Path, workspace_dir: Path,
    if_clone_exists: str, logger: NotebookLogger, clone_depth: int | None = None,
) -> Path:
    if use_git_url:
        return clone_or_reuse_repository(repo_url, workspace_dir, if_clone_exists, logger, clone_depth)
    return validate_local_repo_path(Path(local_repo_path), logger)


def should_exclude_path(path: Path) -> bool:
    return any(part in EXCLUDED_DIR_NAMES for part in path.parts)


def discover_c_files(repo_path: Path) -> list[Path]:
    files: list[Path] = []
    for file_path in repo_path.rglob("*"):
        if not file_path.is_file():
            continue
        if file_path.suffix.lower() not in C_FILE_EXTENSIONS:
            continue
        if should_exclude_path(file_path.relative_to(repo_path)):
            continue
        files.append(file_path.resolve())
    return sorted(files)


def compute_repository_stats(repo_path: Path, c_files: list[Path]) -> dict[str, Any]:
    total_size = sum(path.stat().st_size for path in c_files)
    directories = {path.parent for path in c_files}
    return {
        "repository_name": repo_path.name,
        "repository_size_bytes": total_size,
        "directory_count": len(directories),
        "c_file_count": len(c_files),
    }


def save_c_inventory(c_files: list[Path], output_csv: Path) -> None:
    pd.DataFrame(
        [{"file_path": str(p), "file_name": p.name, "directory": str(p.parent)} for p in c_files]
    ).to_csv(output_csv, index=False)


def download_cloc(cloc_dir: Path) -> Path:
    cloc_dir.mkdir(parents=True, exist_ok=True)
    if sys.platform.startswith("win"):
        target = cloc_dir / "cloc.exe"
        if not target.exists():
            urllib.request.urlretrieve(CLOC_WINDOWS_URL, target)
        return target
    target = cloc_dir / "cloc.pl"
    if not target.exists():
        urllib.request.urlretrieve(CLOC_PERL_URL, target)
    target.chmod(0o755)
    return target


def resolve_cloc_executable(runtimes_root: Path) -> Path:
    env_path = os.environ.get("CLOC")
    if env_path:
        candidate = Path(env_path)
        if candidate.exists():
            return candidate.resolve()
    which = shutil.which("cloc")
    if which:
        return Path(which).resolve()
    runtime_dir = runtimes_root / "cloc"
    for candidate in [runtime_dir / "cloc.exe", runtime_dir / "cloc.pl", runtime_dir / "cloc"]:
        if candidate.exists():
            return candidate.resolve()
    downloaded = download_cloc(runtime_dir)
    return downloaded.resolve()


def build_cloc_command(cloc_exe: Path, repo_path: Path, *, json_output: bool = False, csv_output: bool = False) -> list[str]:
    if cloc_exe.suffix.lower() == ".pl":
        command = ["perl", str(cloc_exe)]
    else:
        command = [str(cloc_exe)]
    command.extend([
        str(repo_path), "--include-lang=C", f"--exclude-dir={CLOC_EXCLUDE_DIRS}", "--quiet",
    ])
    if json_output:
        command.append("--json")
    if csv_output:
        command.append("--csv")
    return command


def run_cloc_command(command: list[str], logger: NotebookLogger) -> tuple[str, str, int]:
    completed = subprocess.run(
        command, capture_output=True, text=True, encoding="utf-8", errors="replace", check=False,
    )
    return completed.stdout, completed.stderr, completed.returncode


def combine_raw_streams(stdout: str, stderr: str) -> str:
    raw = stdout
    if stderr:
        if raw and not raw.endswith("\n"):
            raw += "\n"
        raw += stderr
    return raw


def parse_json_payload(text: str) -> dict[str, Any]:
    if not text.strip():
        return {}
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        return {}
    return payload if isinstance(payload, dict) else {}


def extract_language_metrics(payload: dict[str, Any], language: str = "C") -> dict[str, Any]:
    if language in payload and isinstance(payload[language], dict):
        return payload[language]
    if "SUM" in payload and isinstance(payload["SUM"], dict):
        return payload["SUM"]
    for key, value in payload.items():
        if key == "header" or not isinstance(value, dict):
            continue
        if {"comment", "code"}.issubset(value.keys()):
            return value
    return {}


def parse_cloc_metrics(payload: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for key, value in payload.items():
        if key == "header" or not isinstance(value, dict):
            continue
        if "code" not in value:
            continue
        rows.append({
            "language": key,
            "files": value.get("nFiles", value.get("files", "")),
            "blank_lines": value.get("blank", ""),
            "comment_lines": value.get("comment", ""),
            "code_lines": value.get("code", ""),
        })
    if not rows:
        return pd.DataFrame(columns=METRICS_COLUMNS)
    frame = pd.DataFrame(rows, columns=METRICS_COLUMNS)
    if "SUM" in frame["language"].values:
        frame = frame[frame["language"] != "SUM"].reset_index(drop=True)
    return frame


def compute_comment_metrics(metrics: dict[str, Any]) -> dict[str, float]:
    comment_lines = float(metrics.get("comment", 0) or 0)
    code_lines = float(metrics.get("code", 0) or 0)
    blank_lines = float(metrics.get("blank", 0) or 0)
    files = float(metrics.get("nFiles", metrics.get("files", 0)) or 0)
    ratio = round(comment_lines / code_lines, 4) if code_lines > 0 else 0.0
    percentage = round(ratio * 100, 2)
    return {
        "files": files,
        "blank_lines": blank_lines,
        "comment_lines": comment_lines,
        "code_lines": code_lines,
        "comment_to_code_ratio": ratio,
        "comment_to_code_percentage": percentage,
    }


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW CLOC OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")


## Section 4 — Repository Setup


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / 'error_log.txt'

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)

try:
    REPO_PATH = resolve_repository_path(
        use_git_url=USE_GIT_URL, repo_url=REPO_URL, local_repo_path=LOCAL_REPO_PATH,
        workspace_dir=WORKSPACE_PATH, if_clone_exists=IF_CLONE_EXISTS, logger=logger, clone_depth=CLONE_DEPTH,
    )
except Exception as exc:
    logger.error(f'Repository setup failed: {exc}')
    raise

C_FILES = discover_c_files(REPO_PATH)
if not C_FILES:
    logger.error('No C source files found in repository.', file=str(REPO_PATH))
    raise FileNotFoundError('No C source files found.')

REPO_STATS = compute_repository_stats(REPO_PATH, C_FILES)
CLOC_EXE = resolve_cloc_executable(RUNTIMES_ROOT)
logger.info(f'Repository ready at: {REPO_PATH}')
logger.info(f'Using cloc executable: {CLOC_EXE}')
print(f"Repository: {REPO_STATS['repository_name']}")
print(f"Size (C files): {REPO_STATS['repository_size_bytes']:,} bytes")
print(f"Directories: {REPO_STATS['directory_count']:,}")
print(f"C files: {REPO_STATS['c_file_count']:,}")


## Section 5 — Discover C Files


In [ ]:
INVENTORY_CSV = OUTPUT_PATH / 'c_files_inventory.csv'
save_c_inventory(C_FILES, INVENTORY_CSV)

print(f'Total C Files Found: {len(C_FILES)}')
print(f'Saved inventory to: {INVENTORY_CSV}')


## Section 6 — Execute cloc

Run cloc in text, JSON, and CSV modes. Preserve stdout/stderr exactly as emitted.


In [ ]:
CLOC_CONSOLE_CHUNKS: list[str] = []
CLOC_JSON = ''
CLOC_CSV = ''

for label, json_output, csv_output in [('text', False, False), ('json', True, False), ('csv', False, True)]:
    cmd = build_cloc_command(CLOC_EXE, REPO_PATH, json_output=json_output, csv_output=csv_output)
    stdout, stderr, code = run_cloc_command(cmd, logger)
    CLOC_CONSOLE_CHUNKS.append(f'===== cloc ({label}) =====\n' + combine_raw_streams(stdout, stderr))
    if code != 0:
        logger.error(f'cloc {label} run exited with code {code}', file=f'cloc_{label}')
    if json_output:
        CLOC_JSON = stdout
    elif csv_output:
        CLOC_CSV = stdout

logger.info('cloc execution complete.')


## Section 7 — Raw Output Extraction


In [ ]:
CONSOLE_PATH = OUTPUT_PATH / 'cloc_raw_console_output.txt'
JSON_PATH = OUTPUT_PATH / 'cloc_output.json'
CSV_PATH = OUTPUT_PATH / 'cloc_output.csv'

CONSOLE_PATH.write_text('\n'.join(CLOC_CONSOLE_CHUNKS), encoding='utf-8')
JSON_PATH.write_text(CLOC_JSON, encoding='utf-8')
CSV_PATH.write_text(CLOC_CSV, encoding='utf-8')

logger.info('Saved cloc raw console, JSON, and CSV outputs.')
preview_raw_output(CONSOLE_PATH.read_text(encoding='utf-8'), RAW_OUTPUT_PREVIEW_LINES, CONSOLE_PATH)


## Section 8 — Parse cloc Output


In [ ]:
JSON_PAYLOAD = parse_json_payload(CLOC_JSON)
METRICS_DF = parse_cloc_metrics(JSON_PAYLOAD)
METRICS_CSV = OUTPUT_PATH / 'cloc_metrics.csv'
METRICS_DF.to_csv(METRICS_CSV, index=False)

logger.info(f'Parsed cloc metrics rows={len(METRICS_DF)}')
display(METRICS_DF)


## Section 9 — Comment-to-Code Ratio (Derived)

**Derived metric** (not emitted directly by cloc):

```text
Total_Comment_Lines = comment
Comment_to_Code_Ratio = comment / code
```


In [ ]:
C_METRICS = extract_language_metrics(JSON_PAYLOAD, 'C')
COMMENT_METRICS = compute_comment_metrics(C_METRICS)
RATIO_CSV = OUTPUT_PATH / 'comment_to_code_ratio_summary.csv'
pd.DataFrame([
    {'metric_name': 'Comment_to_Code_Ratio', 'metric_value': COMMENT_METRICS['comment_to_code_ratio']},
]).to_csv(RATIO_CSV, index=False)

logger.info(
    f"Comment-to-Code Ratio={COMMENT_METRICS['comment_to_code_ratio']} "
    f"(Derived from cloc metrics)"
)
display(pd.read_csv(RATIO_CSV))


## Section 10 — Comment Percentage (Derived)


In [ ]:
PERCENTAGE_CSV = OUTPUT_PATH / 'comment_percentage_summary.csv'
pd.DataFrame([
    {'metric_name': 'Comment_to_Code_Percentage', 'metric_value': COMMENT_METRICS['comment_to_code_percentage']},
]).to_csv(PERCENTAGE_CSV, index=False)

logger.info(f"Comment Percentage={COMMENT_METRICS['comment_to_code_percentage']}%")
display(pd.read_csv(PERCENTAGE_CSV))


## Section 11 — Summary Dashboard


In [ ]:
summary_df = pd.DataFrame([
    {'Metric': 'Total C Files', 'Value': len(C_FILES)},
    {'Metric': 'Total Blank Lines', 'Value': int(COMMENT_METRICS['blank_lines'])},
    {'Metric': 'Total Comment Lines', 'Value': int(COMMENT_METRICS['comment_lines'])},
    {'Metric': 'Total Code Lines', 'Value': int(COMMENT_METRICS['code_lines'])},
    {'Metric': 'Comment-to-Code Ratio (Derived)', 'Value': COMMENT_METRICS['comment_to_code_ratio']},
    {'Metric': 'Comment Percentage (Derived)', 'Value': COMMENT_METRICS['comment_to_code_percentage']},
])
display(summary_df)

deliverables = [
    CONSOLE_PATH, JSON_PATH, CSV_PATH, METRICS_CSV, RATIO_CSV, PERCENTAGE_CSV,
    INVENTORY_CSV, ERROR_LOG_PATH,
]
print('\nDeliverables:')
for path in deliverables:
    print(f"  [{'OK' if path.exists() else 'MISSING'}] {path}")


## Section 12 — Error Handling


In [ ]:
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size > 0:
    print(ERROR_LOG_PATH.read_text(encoding='utf-8'))
else:
    print('No errors logged.')


## Section 13 — Deliverables

```text
outputs/
├── cloc_raw_console_output.txt
├── cloc_output.json
├── cloc_output.csv
├── cloc_metrics.csv
├── comment_to_code_ratio_summary.csv
├── comment_percentage_summary.csv
├── c_files_inventory.csv
└── error_log.txt
```
